In [1]:
import pandas as pd
import numpy as np

# Load the cleaned dataset we created in Step 3
df = pd.read_csv('../data/cleaned_rideshare.csv')

# Verify the data loaded correctly
print("Data shape after loading:", df.shape)
df.head()

Data shape after loading: (286457, 57)


,id,timestamp,hour,day,month,datetime,timezone,source,destination,cab_type,...,precipIntensityMax,uvIndexTime,temperatureMin,temperatureMinTime,temperatureMax,temperatureMaxTime,apparentTemperatureMin,apparentTemperatureMinTime,apparentTemperatureMax,apparentTemperatureMaxTime
0,4bd23055-6827-41c6-b23b-3c491f24e74d,1.543284e+09,2,27,11,2018-11-27 02:00:23,America/New_York,Haymarket Square,North Station,Lyft,...,0.1300,1543251600,40.49,1543233600,47.30,1543251600,36.20,1543291200,43.92,1543251600
1,981a3613-77af-4620-a42a-0c0866077d1e,1.543367e+09,1,28,11,2018-11-28 01:00:22,America/New_York,Haymarket Square,North Station,Lyft,...,0.1064,1543338000,35.36,1543377600,47.55,1543320000,31.04,1543377600,44.12,1543320000
2,c2d88af2-d278-4bfd-a8d0-29ca77cc5512,1.543554e+09,4,30,11,2018-11-30 04:53:02,America/New_York,Haymarket Square,North Station,Lyft,...,0.0000,1543507200,34.67,1543550400,45.03,1543510800,30.30,1543550400,38.53,1543510800
3,e0126e1f-8ca9-4f2e-82b3-50505a09db9a,1.543463e+09,3,29,11,2018-11-29 03:49:20,America/New_York,Haymarket Square,North Station,Lyft,...,0.0001,1543420800,33.10,1543402800,42.18,1543420800,29.11,1543392000,35.75,1543420800
4,462816a3-820d-408b-8549-0b39e82f65ac,1.543209e+09,5,26,11,2018-11-26 05:03:00,America/New_York,Back Bay,Northeastern University,Lyft,...,0.1245,1543251600,40.67,1543233600,46.46,1543255200,37.45,1543291200,43.81,1543251600


In [2]:
# 1. Driver Gross Earnings: Total money paid for the ride
# Formula: Base price + tips/bonuses (if any, using standard ride price here)
df['driver_gross'] = df['price']

# 2. Driver Net Earnings: Company (PickMe/Uber) takes 20% commission, driver keeps 80%
df['driver_net'] = df['driver_gross'] * 0.80

# 3. Driver Net Earnings Per Hour: Estimated based on distance (assuming average city speed 25 mph)
# Ride Duration in Hours = (distance / 25)
df['ride_duration_hours'] = df['distance'] / 25.0
df['driver_net_per_hour'] = df['driver_net'] / df['ride_duration_hours']

# Preview the new earnings data
df[['price', 'driver_gross', 'driver_net', 'driver_net_per_hour']].head()

,price,driver_gross,driver_net,driver_net_per_hour
0,11.0,11.0,8.8,500.000000
1,7.0,7.0,5.6,318.181818
2,26.0,26.0,20.8,1181.818182
3,9.0,9.0,7.2,409.090909
4,10.5,10.5,8.4,194.444444


In [3]:
# Create a binary flag: 1 if driver net per hour is below $8, else 0
df['low_pay_flag'] = np.where(df['driver_net_per_hour'] < 8.0, 1, 0)

# Check how many rides are flagged as low pay
print(df['low_pay_flag'].value_counts())

low_pay_flag
0    286457
Name: count, dtype: int64


In [4]:
# Make sure datetime column is parsed correctly
df['datetime'] = pd.to_datetime(df['datetime'])

# Extract useful components from the datetime column
df['hour_of_day'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek  # 0 = Monday, 6 = Sunday

# Create Weekend Flag: 1 if Saturday (5) or Sunday (6), else 0
df['is_weekend'] = np.where(df['day_of_week'].isin([5, 6]), 1, 0)

# Verify the new time features
df[['datetime', 'hour_of_day', 'day_of_week', 'is_weekend']].head()

,datetime,hour_of_day,day_of_week,is_weekend
0,2018-11-27 02:00:23,2,1,0
1,2018-11-28 01:00:22,1,2,0
2,2018-11-30 04:53:02,4,4,0
3,2018-11-29 03:49:20,3,3,0
4,2018-11-26 05:03:00,5,0,0


In [5]:
# Save this feature-rich dataset for EDA and Modeling steps
df.to_csv('../data/engineered_rideshare.csv', index=False)

print("Success: Engineered dataset saved as engineered_rideshare.csv!")

Success: Engineered dataset saved as engineered_rideshare.csv!
